# 第08章 B2组（G18）独立验证

受审对象：A2/G17，commit `8a730325416ca80af533258229b22ad3461e855f`，
`教材/第08章-加速的秘诀/工作/A2-G17-实验.ipynb`。
本文件是 B2 的独立对照计算；A2 的原 Notebook 已在单独的 Python 3.12 环境从头运行，
运行证据见 `A2-8a730325-复现.ipynb`。这里的参考值来自解析式、已知解或独立构造，
不是直接把 A2 的输出当作标准答案。

披露：本文件是在查看 A2 实现之后编制，不能称为查看 A 前的盲审推导。


In [1]:
import platform
import sys
import numpy as np
import scipy

REVIEW_SHA = "8a730325416ca80af533258229b22ad3461e855f"
print("B2/G18 | A2 SHA:", REVIEW_SHA)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "SciPy:", scipy.__version__)
print("系统:", platform.platform())
print("内核:", sys.executable)
assert sys.version_info[:2] == (3, 12)
assert sys.prefix != sys.base_prefix, "请在独立虚拟环境的内核中运行"


B2/G18 | A2 SHA: 8a730325416ca80af533258229b22ad3461e855f
Python: 3.12.4 NumPy: 1.26.4 SciPy: 1.14.1
系统: Windows-11-10.0.26200-SP0
内核: C:\Users\111\Documents\Codex\2026-11-08\06-b-md-https-github-com\work\env-a1\Scripts\python.exe


## 8.1 正定性与已知解

参考输入：16×16 网格，严格正的空间系数和零边界。五点通量离散的能量
由正权边差平方与边界项组成，因此矩阵应正定。另取已知向量 `x_true`，
用 `b=A@x_true` 检查直接解。对本独立样例，容差为相对误差 `<1e-10`。
A2 的 40 个随机矩阵仍以 A2 固定版本的运行记录为证据。


In [2]:
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh, spsolve

side = 16
grid = np.arange(side * side).reshape(side, side)
u, v = np.meshgrid(np.linspace(0, 1, side), np.linspace(0, 1, side))
k = 1 + 0.5 * np.sin(np.pi * u)**2 * np.cos(np.pi * v)**2
A = lil_matrix((side * side, side * side), dtype=float)
for i in range(side):
    for j in range(side):
        p = grid[i, j]
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ii, jj = i + di, j + dj
            if 0 <= ii < side and 0 <= jj < side:
                weight = 2 * k[i, j] * k[ii, jj] / (k[i, j] + k[ii, jj])
                A[p, grid[ii, jj]] -= weight
            else:
                weight = k[i, j]
            A[p, p] += weight
A = A.tocsr()
min_eigenvalue = float(eigsh(A, k=1, which="SA", return_eigenvectors=False)[0])
x_true = np.sin(np.arange(1, side * side + 1))
x_computed = spsolve(A, A @ x_true)
relative_error = np.linalg.norm(x_computed - x_true) / np.linalg.norm(x_true)
print(f"最小特征值={min_eigenvalue:.9f}; 已知解相对误差={relative_error:.3e}")
assert min_eigenvalue > 0
assert relative_error < 1e-10


最小特征值=0.078237428; 已知解相对误差=1.168e-15


## 8.2 两模态快照的 POD

参考输入与 A2 相同：`sin(pi*x) exp(-t) + 0.2 sin(2*pi*x) cos(t)`；
按前 21 时刻训练、后 9 时刻测试。解析上数据至多秩 2，故保留 2 个
POD 模态应达到浮点误差级。容差：测试相对 Frobenius 误差 `<1e-12`。


In [3]:
space = np.linspace(0, 1, 80)
times = np.linspace(0, 2, 30)
snapshots = (np.sin(np.pi * space)[:, None] * np.exp(-times)[None, :]
             + 0.2 * np.sin(2 * np.pi * space)[:, None] * np.cos(times)[None, :])
train, test = snapshots[:, :21], snapshots[:, 21:]
mean = train.mean(axis=1, keepdims=True)
left, singular_values, _ = np.linalg.svd(train - mean, full_matrices=False)
def pod_error(rank):
    restored = mean + left[:, :rank] @ (left[:, :rank].T @ (test - mean))
    return np.linalg.norm(test - restored) / np.linalg.norm(test)
err_rank1, err_rank2 = pod_error(1), pod_error(2)
print(f"测试误差 r=1: {err_rank1:.6f}; r=2: {err_rank2:.3e}")
assert abs(err_rank1 - 0.502) < 0.002
assert err_rank2 < 1e-12


测试误差 r=1: 0.502341; r=2: 1.048e-15


## 8.3 解析 AR(3) 参考

指数 `exp(-t)` 与余弦 `cos(t)` 在等步长采样时满足特征根
`exp(-dt), exp(i*dt), exp(-i*dt)` 的三阶递推。用解析递推从训练段
末尾自由滚动 40 步，与解析真值比较；容差 `<1e-8`。
这核验 A2 中 AR(3) 近零误差的可解释性，不代表其 MLP 能泛化。


In [4]:
dt = 0.05
t = np.arange(121) * dt
rootsum = np.exp(-dt) + 2 * np.cos(dt)
pairsum = 1 + 2 * np.exp(-dt) * np.cos(dt)
rootproduct = np.exp(-dt)
truth = np.vstack((np.exp(-t), np.cos(t)))
forecast = truth[:, :81].copy()
for _ in range(40):
    next_state = (rootsum * forecast[:, -1] - pairsum * forecast[:, -2]
                  + rootproduct * forecast[:, -3])
    forecast = np.column_stack((forecast, next_state))
rolling_error = np.sqrt(np.mean((forecast[:, 81:] - truth[:, 81:])**2))
print(f"解析 AR(3) 自由滚动 RMSE={rolling_error:.3e}")
assert rolling_error < 1e-8


解析 AR(3) 自由滚动 RMSE=1.517e-13


## 8.4 解析开环与有噪声闭环

参考真值为频率 1 的旋转，模型频率 1.15、初值 `[1.05,-0.05]`。
开环终点可直接用旋转解析式算出；闭环用独立的标量递推和种子 2026
重算观测间隔 5、增益 0.7 的结果。与 A2 报告到 3 位小数的数值比对，
容差为 `0.005`。


In [5]:
steps = 400
dt = 0.05
t = np.arange(steps) * dt
truth = np.column_stack((np.cos(t), np.sin(t)))
phase = 1.15 * t[-1]
initial = np.array([1.05, -0.05])
open_end = np.array([initial[0] * np.cos(phase) - initial[1] * np.sin(phase),
                     initial[0] * np.sin(phase) + initial[1] * np.cos(phase)])
open_error = np.linalg.norm(open_end - truth[-1])

rng = np.random.default_rng(2026)
c, s = np.cos(1.15 * dt), np.sin(1.15 * dt)
state = initial.copy()
for index in range(1, steps):
    state = np.array([c * state[0] - s * state[1],
                      s * state[0] + c * state[1]])
    if index % 5 == 0:
        observed = np.cos(t[index]) + rng.normal(0, 0.02)
        state[0] += 0.7 * (observed - state[0])
closed_error = np.linalg.norm(state - truth[-1])
print(f"开环终点误差={open_error:.6f}; 闭环终点误差={closed_error:.6f}")
assert abs(open_error - 2.041) < 0.005
assert abs(closed_error - 0.184) < 0.005


开环终点误差=2.041286; 闭环终点误差=0.184314


## 核验边界

上述四节参考值与 A2 在独立内核的保存输出一致。A2 8.1 的动态计时和
自编码器、MLP 的训练表现依赖设备、数值库及随机状态，不以单次数值做
严格常数断言。8.4 的频率扫描每次重新抽噪声，不能把不同间隔的单次
终点误差解释为仅由更新间隔造成的差异。正式报告列出修正项。
